# CLAMPfull pseudobulk disentanglement proof

Checks that CLAMPfull's latent variables (LVs) disentangle cell types: each cell
type correlates with one distinct LV (from `00_benchmark`), and that LV's
top-loading genes independently recover markers of the same cell type (via the
CellMarker database).

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
library(here)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(stringr)
library(readxl)
source(here("scripts", "pseudobulk", "common.R"))

## Settings

In [ ]:
DATASETS     <- c("Brain_Mathys2023", "Brain_Xiong2023",
                  "PBMC_1k1k", "PBMC_Perez2022",
                  "Heart_Datar2026", "Lung_Sikkema2023")
MOD_ROOT     <- here("output", "01_model_building", "05_pseudobulk")
DATA_ROOT    <- here("data", "pseudobulk")
BENCH_DIR    <- here("output", "03_model_biology", "02_pseudobulk", "00_benchmark")
OUT_DIR      <- here("output", "03_model_biology", "02_pseudobulk", "02_disentangle")
TOP_GENE_PCT <- 0.01     # top 1% gene loadings for ORA

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
set.seed(42)

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"),
                         stringsAsFactors = FALSE)
CT_LABELS    <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label     <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

## Each cell type correlates with a distinct LV

In [ ]:
ASSIGNMENTS_PATH <- file.path(BENCH_DIR, "clampfull_lv_assignments.csv")

top_lvs_df <- read.csv(ASSIGNMENTS_PATH, stringsAsFactors = FALSE)

# LUAD excluded via DATASETS filter
top_lvs_df <- top_lvs_df[top_lvs_df$dataset %in% DATASETS, ]
rownames(top_lvs_df) <- NULL

cat("CLAMPfull LV assignments loaded:", nrow(top_lvs_df), "rows\n")
cat("Datasets:\n")
print(table(top_lvs_df$dataset))
cat("\nSample (first 10 rows):\n")
print(head(top_lvs_df, 10))

## Different cell types are captured by different LVs

In [ ]:
CORR_FULL_PATH <- file.path(BENCH_DIR, "clampfull_lv_ct_corr_full.csv")
lv_corr_full   <- read.csv(CORR_FULL_PATH, stringsAsFactors = FALSE)
lv_corr_full   <- lv_corr_full[lv_corr_full$dataset %in% DATASETS, ]

cat("Full LV x cell-type correlation matrix loaded:", nrow(lv_corr_full), "rows\n")

# Use the same readable cell-type names as the marker-recovery dot plot.
# Keep the number of columns/rows so each dataset can be rendered with
# consistently sized correlation tiles.
heatmap_panels <- lapply(DATASETS, function(ds) {
    d <- lv_corr_full[lv_corr_full$dataset == ds, ]
    if (nrow(d) == 0) return(NULL)

    # Skin contains an author-provided missing category (the literal "NA");
    # it is not a cell type and should not be represented in the heatmap.
    d <- d[!is.na(d$cell_type) & d$cell_type != "NA", ]
    assigned <- top_lvs_df[top_lvs_df$dataset == ds, c("cell_type", "LV")]
    assigned <- assigned[!is.na(assigned$cell_type) & assigned$cell_type != "NA", ]
    if (nrow(d) == 0 || nrow(assigned) == 0) return(NULL)
    # Restrict both axes to the selected cell types. This matches the dot
    # plot and prevents unselected Skin cell types from collapsing into an
    # artificial NA factor level.
    d <- d[d$LV %in% assigned$LV & d$cell_type %in% assigned$cell_type, ]
    d$cell_type_label <- ct_label(d$cell_type)

    ord      <- order(assigned$cell_type)
    lv_order <- unique(assigned$LV[ord])
    ct_order <- unique(ct_label(assigned$cell_type[ord]))
    d$LV              <- factor(d$LV, levels = lv_order)
    d$cell_type_label <- factor(d$cell_type_label, levels = ct_order)

    p <- ggplot(d, aes(x = cell_type_label, y = LV, fill = cor)) +
        geom_tile(color = "white", linewidth = 0.4) +
        geom_text(aes(label = sprintf("%.2f", cor)), size = 2.5) +
        # Keep weak-to-moderate correlations visually close to white.
        scale_fill_gradientn(colours = c("#b2182b", "#fdf7f7", "white", "#f7fbf7", "#007a33"),
                             values = scales::rescale(c(-1, -0.5, 0, 0.5, 1)),
                             limits = c(-1, 1), name = "r") +
        coord_fixed() +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = ds)

    list(dataset = ds, plot = p, n_cell_types = length(ct_order), n_lvs = length(lv_order))
})
heatmap_panels <- Filter(Negate(is.null), heatmap_panels)

# Render and save one figure per dataset. The saved PNG dimensions are
# calculated from the grid, so more cell types/LVs make a genuinely larger
# image rather than merely stretching to the notebook viewer.
HEATMAP_TILE_SIZE <- 0.55  # inches per correlation tile
HEATMAP_PLOT_DIR <- file.path(OUT_DIR, "plots", "lv_cell_type_correlations")
dir.create(HEATMAP_PLOT_DIR, recursive = TRUE, showWarnings = FALSE)
for (panel in heatmap_panels) {
    plot_width  <- 3.0 + HEATMAP_TILE_SIZE * panel$n_cell_types
    plot_height <- 2.5 + HEATMAP_TILE_SIZE * panel$n_lvs
    p <- panel$plot + theme(legend.position = "right")
    ggsave(
        filename = file.path(HEATMAP_PLOT_DIR, paste0(panel$dataset, "_lv_cell_type_correlation.png")),
        plot = p, width = plot_width, height = plot_height, units = "in",
        dpi = 100, limitsize = FALSE
    )
    # Display the already-sized raster. Printing p here would reopen the
    # notebook graphics device and stretch every dataset independently.
    IRdisplay::display_png(file = file.path(
        HEATMAP_PLOT_DIR, paste0(panel$dataset, "_lv_cell_type_correlation.png")
    ))
}

## Module-level marker recovery — top-loading genes recover independent cell-type markers

In [ ]:
CM_PATH <- here("data", "pathways", "Cell_marker_Human.xlsx")

load_cellmarker_xlsx <- function(path, sheet, species_keep = "Human") {
    df <- readxl::read_excel(path, sheet = sheet)
    df <- df[df$species %in% species_keep & !is.na(df$Symbol) & nzchar(df$Symbol), ]
    split(df$Symbol, df$cell_name)
}
cm_t2g <- load_cellmarker_xlsx(CM_PATH, "human")

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub("[^a-z0-9]+", " ", x)
    stringr::str_squish(x)
}

# Marker sets: CellMarker (Cell_marker_Human.xlsx) terms matched to each cell
# type by alias, with a promiscuity filter dropping genes shared by >50% of a
# dataset's marker sets (pan-lineage genes like PTPRC/CD3E).
alias_map <- list(
    Ast = c("astrocyte"),
    Exc = c("excitatory neuron", "glutamatergic neuron", "glutamatergic cell"),
    Inh = c("inhibitory neuron", "gabaergic neuron", "gabaergic cell"),
    Mic = c("microglia", "microglial"),
    Oli = c("oligodendrocyte"),
    Opc = c("oligodendrocyte precursor", "oligodendrocyte progenitor", "opc"),
    Vas = c("vascular", "endothelial cell", "pericyte", "smooth muscle cell"),
    B_cell = c("b cell", "b lymphocyte"),
    CD14_Mono = c("cd14 monocyte", "cd14 positive monocyte", "classical monocyte", "monocyte"),
    CD16_Mono = c("cd16 monocyte", "cd16 positive monocyte", "non classical monocyte", "nonclassical monocyte", "monocyte"),
    CD4_T = c("cd4 t cell", "cd4 positive t cell", "cd4 lymphocyte"),
    CD8_T = c("cd8 t cell", "cd8 positive t cell", "cd8 lymphocyte", "cd8"),
    DC = c("dendritic cell"),
    gd_T = c("gamma delta t cell", "gd t cell", "gammadelta t cell"),
    NK = c("natural killer", "nk cell"),
    Plasma_B = c("plasma cell", "plasma b cell", "antibody secreting cell"),
    T_cell = c("t cell", "t lymphocyte"),
    Myeloid = c("myeloid", "monocyte", "macrophage"),
    Myeloid_cell = c("myeloid", "monocyte", "macrophage"),
    Dendritic = c("dendritic cell"),
    Endothelial = c("endothelial cell"),
    Endothelial_cell = c("endothelial cell"),
    Fibroblast = c("fibroblast"),
    Malignant = c("malignant", "cancer cell", "tumor cell", "carcinoma"),
    Mast = c("mast cell"),
    Mast_cell = c("mast cell"),
    Cancer_cell = c("cancer cell", "malignant", "tumor cell", "carcinoma"),
    pDC = c("plasmacytoid dendritic", "pdc")
)


# Additional broad labels for temporary Heart/Lung/PanCancer pseudobulk datasets.
alias_map <- c(alias_map, list(
    VentricularCM = c("ventricular cardiomyocyte", "cardiac myocyte", "cardiomyocyte"),
    AtrialCM = c("atrial cardiomyocyte", "cardiac myocyte", "cardiomyocyte"),
    Endocardial = c("endocardial cell", "endothelium", "endothelial cell"),
    Epicardial = c("epicardial cell", "mesothelial cell"),
    LymphaticEndothelial = c("lymphatic endothelial cell", "lymphatic endothelium"),
    Lymphocyte = c("lymphocyte", "t cell", "b cell", "natural killer"),
    Macrophage = c("macrophage", "myeloid"),
    Neuronal = c("neuron", "neuronal"),
    Pericyte = c("pericyte"),
    VSMC = c("vascular smooth muscle", "smooth muscle cell"),
    Adipocyte = c("adipocyte"),
    # Lung_Sikkema2023 ann_level_3 fine-type aliases below are unused now that
    # truthFrac_v0 uses ann_level_2 (coarse aliases added further down) --
    # kept only as reference for ann_finest_level (truthFrac_v1) work.
    AT1 = c("alveolar type 1", "type i pneumocyte", "at1 cell"),
    AT2 = c("alveolar type 2", "type ii pneumocyte", "at2 cell"),
    `B cell lineage` = c("b cell", "b lymphocyte"),
    Basal = c("basal cell"),
    `Dendritic cells` = c("dendritic cell"),
    `EC arterial` = c("arterial endothelial", "artery endothelial", "endothelial cell"),
    `EC capillary` = c("capillary endothelial", "endothelial cell"),
    `EC venous` = c("venous endothelial", "vein endothelial", "endothelial cell"),
    Fibroblasts = c("fibroblast"),
    `Innate lymphoid cell NK` = c("natural killer", "innate lymphoid", "nk cell"),
    `Lymphatic EC differentiating` = c("lymphatic endothelial"),
    `Lymphatic EC mature` = c("lymphatic endothelial"),
    `Lymphatic EC proliferating` = c("lymphatic endothelial"),
    Macrophages = c("macrophage"),
    `Mast cells` = c("mast cell"),
    Monocytes = c("monocyte"),
    `Multiciliated lineage` = c("ciliated cell", "multiciliated cell"),
    Myofibroblasts = c("myofibroblast"),
    `SM activated stress response` = c("smooth muscle"),
    Secretory = c("secretory cell", "club cell", "goblet cell"),
    `Smooth muscle FAM83D+` = c("smooth muscle"),
    `Submucosal Secretory` = c("submucosal gland", "serous cell", "mucous cell"),
    `T cell lineage` = c("t cell", "t lymphocyte"),
    Epithelial = c("epithelial", "epithelium", "carcinoma", "tumor cell", "malignant"),
    `T cell` = c("t cell", "t lymphocyte"),
    `B cell` = c("b cell", "b lymphocyte"),
    `NK cell` = c("natural killer", "nk cell"),
    `Dendritic cell` = c("dendritic cell"),
    Fibroblast = c("fibroblast"),
    Endothelial = c("endothelial cell"),
    Malignant = c("malignant", "cancer cell", "tumor cell", "carcinoma")
))

# Lung_Sikkema2023 ann_level_2 coarse-compartment aliases (truthFrac_v0's
# actual truth columns). HLCA compartment names like "Blood vessels" or
# "Airway epithelium" aren't themselves CellMarker cell-type entries, so
# without these they'd match 0 marker genes and drop out of the ORA panel
# entirely (only "Myeloid"/"Lymphoid" survived via generic substrings).
alias_map <- c(alias_map, list(
    `Airway epithelium` = c("ciliated cell", "club cell", "goblet cell", "basal cell"),
    `Alveolar epithelium` = c("alveolar epithelial cell", "type i pneumocyte", "type ii pneumocyte"),
    `Blood vessels` = c("endothelial cell", "vascular endothelial cell"),
    `Fibroblast lineage` = c("fibroblast", "myofibroblast"),
    `Lymphatic EC` = c("lymphatic endothelial cell", "lymphatic endothelium"),
    `Submucosal Gland` = c("submucosal gland", "serous cell", "mucous cell")
))


match_aliases <- function(cell_type, terms) {
    aliases <- alias_map[[cell_type]]
    if (is.null(aliases)) aliases <- normalize_text(ct_label(cell_type))
    aliases_norm <- normalize_text(aliases)
    terms_norm   <- normalize_text(terms)
    vapply(terms_norm, function(term) {
        any(vapply(aliases_norm, function(alias) stringr::str_detect(term, stringr::fixed(alias)), logical(1)))
    }, logical(1))
}

build_marker_sets <- function(cell_types, gene_universe, min_size = 5, promiscuity_max_frac = 0.5) {
    cm_terms <- names(cm_t2g)
    sets <- lapply(cell_types, function(ct) {
        hit_sets <- cm_terms[match_aliases(ct, cm_terms)]
        genes    <- unique(unlist(cm_t2g[hit_sets]))
        intersect(genes, gene_universe)
    })
    names(sets) <- cell_types
    sets <- Filter(function(g) length(g) >= min_size, sets)

    if (length(sets) > 1) {
        gene_counts <- table(unlist(sets))
        promiscuous <- names(gene_counts)[gene_counts > promiscuity_max_frac * length(sets)]
        sets <- lapply(sets, function(g) setdiff(g, promiscuous))
        sets <- Filter(function(g) length(g) >= min_size, sets)
    }
    sets
}

read_B_matrix <- function(path) {
    df  <- read.csv(path, row.names = 1, check.names = FALSE)
    mat <- as.matrix(df)
    storage.mode(mat) <- "numeric"
    t(mat)   # samples x LVs
}

assign_dominant_ct <- function(truth_path) {
    truth    <- read.csv(truth_path, row.names = 1, check.names = FALSE)
    ct_names <- colnames(truth)
    dom      <- ct_names[apply(truth, 1, which.max)]
    setNames(dom, rownames(truth))
}

cat("CellMarker (Human) gene sets available for matching:", length(cm_t2g), "\n")

In [ ]:
# One-vs-rest Wilcoxon test per LV per cell type, then a one-to-one greedy
# assignment (like 00_benchmark's oneToOneMask) so no LV is reused within a
# dataset. Cell types never dominant enough to test fall back to Property 2's
# correlation-based assignment (top_lvs_df); fallback LVs are reserved first so
# the two selection paths can't collide.
one_vs_rest_test <- function(B, sample_ct) {
    shared <- intersect(rownames(B), names(sample_ct))
    B  <- B[shared, , drop = FALSE]
    ct <- sample_ct[shared]

    out <- do.call(rbind, lapply(unique(ct), function(this_ct) {
        in_idx <- ct == this_ct
        do.call(rbind, lapply(colnames(B), function(lv) {
            x_in  <- B[in_idx,  lv]
            x_out <- B[!in_idx, lv]
            if (length(x_in) < 3 || length(x_out) < 3) return(NULL)
            wt <- suppressWarnings(wilcox.test(x_in, x_out, alternative = "greater"))
            data.frame(
                cell_type = this_ct, LV = lv,
                n_in = length(x_in), n_out = length(x_out),
                mean_in = mean(x_in), mean_out = mean(x_out),
                effect  = mean(x_in) - mean(x_out),
                pvalue  = wt$p.value,
                stringsAsFactors = FALSE
            )
        }))
    }))
    out$qvalue <- p.adjust(out$pvalue, method = "BH")
    out
}

select_top_lv <- function(ovr_df, qval_threshold = 0.05, reserved_lvs = character(0)) {
    d <- ovr_df[!(ovr_df$LV %in% reserved_lvs), ]
    if (nrow(d) == 0) return(ovr_df[0, ])
    d$is_significant <- d$qvalue < qval_threshold & d$effect > 0
    d <- d[order(-d$is_significant, -d$effect), ]

    assigned <- list()
    used_lvs <- character(0)
    for (i in seq_len(nrow(d))) {
        row <- d[i, ]
        if (row$cell_type %in% names(assigned) || row$LV %in% used_lvs) next
        row$weak_evidence <- !row$is_significant
        assigned[[row$cell_type]] <- row
        used_lvs <- c(used_lvs, row$LV)
    }
    if (length(assigned) == 0) return(ovr_df[0, ])
    do.call(rbind, assigned)
}

run_module_pipeline <- function(ds, method = "CLAMPfull") {
    b_path     <- file.path(MOD_ROOT, ds, method, "B.csv")
    z_path     <- file.path(MOD_ROOT, ds, method, "Z.csv")
    truth_path <- file.path(DATA_ROOT, ds, "truthFrac_v0.csv")
    if (!all(file.exists(b_path), file.exists(z_path), file.exists(truth_path))) return(NULL)

    B         <- read_B_matrix(b_path)
    Z         <- read.csv(z_path, row.names = 1, check.names = FALSE)
    truth <- read.csv(truth_path, row.names = 1, check.names = FALSE)
    # For studies with <=100 samples, exclude cell types below 0.5% mean prevalence.
    if (nrow(truth) <= 100) {
        keep <- colMeans(truth, na.rm = TRUE) >= 0.005
        truth <- truth[, keep, drop = FALSE]
    }
    sample_ct <- setNames(colnames(truth)[apply(truth, 1, which.max)], rownames(truth))
    all_cts   <- colnames(truth)

    universe    <- rownames(Z)
    marker_sets <- build_marker_sets(all_cts, universe)

    ovr <- one_vs_rest_test(B, sample_ct)
    ovr <- ovr[ovr$cell_type %in% names(marker_sets), ]

    sel_fallback <- NULL
    if (method == "CLAMPfull") {
        missing_cts <- setdiff(names(marker_sets), unique(ovr$cell_type))
        fallback <- top_lvs_df[top_lvs_df$dataset == ds & top_lvs_df$cell_type %in% missing_cts, ]
        if (nrow(fallback) > 0) sel_fallback <- data.frame(
            cell_type = fallback$cell_type, LV = fallback$LV,
            n_in = NA_real_, n_out = NA_real_, mean_in = NA_real_, mean_out = NA_real_,
            effect = fallback$cor, pvalue = NA_real_, qvalue = NA_real_,
            is_significant = FALSE, weak_evidence = TRUE,
            selection_method = "correlation_fallback", stringsAsFactors = FALSE
        )
    }
    fallback_lvs <- if (!is.null(sel_fallback)) sel_fallback$LV else character(0)

    sel_ovr <- if (nrow(ovr) > 0) select_top_lv(ovr, reserved_lvs = fallback_lvs) else
        data.frame(cell_type = character(), LV = character(), effect = numeric(),
                  qvalue = numeric(), is_significant = logical(), weak_evidence = logical(),
                  stringsAsFactors = FALSE)
    if (nrow(sel_ovr) > 0) sel_ovr$selection_method <- "one_vs_rest"

    sel <- dplyr::bind_rows(sel_ovr, sel_fallback)

    list(dataset = ds, method = method, ovr = ovr, selected = sel,
         marker_sets = marker_sets, universe = universe)
}

module_results <- lapply(DATASETS, run_module_pipeline)
names(module_results) <- DATASETS
module_results <- Filter(Negate(is.null), module_results)

ovr_all_df <- dplyr::bind_rows(lapply(names(module_results), function(ds)
    cbind(dataset = ds, module_results[[ds]]$ovr)))
selected_all_df <- dplyr::bind_rows(lapply(names(module_results), function(ds)
    cbind(dataset = ds, module_results[[ds]]$selected)))

cat("One-vs-rest tests:", nrow(ovr_all_df), "\n")
cat("Selected LVs (one per cell type per dataset, all cell types covered):", nrow(selected_all_df), "\n")
print(table(selected_all_df$selection_method))
print(selected_all_df[, c("dataset", "cell_type", "LV", "effect", "selection_method", "is_significant")])

In [ ]:
coverage_df <- selected_all_df[, c("dataset", "cell_type", "selection_method")]

cat("Selection method breakdown:\n")
print(table(coverage_df$selection_method))
cat("\nCell types selected via correlation fallback (no one-vs-rest test possible):\n")
print(coverage_df[coverage_df$selection_method == "correlation_fallback",
                  c("dataset", "cell_type")])

In [ ]:
# ORA per selected LV against the full marker database -- top 1% loading genes
# (TOP_GENE_PCT) tested via clusterProfiler::enricher(). Reports up to the top
# 20 significant pathways (FDR < 0.05, fewer than 20 if fewer clear it). If
# NOTHING clears FDR < 0.05 at 1%, retries once at TOP_GENE_PCT_FALLBACK (3%)
# before giving up (NA). "recovered" = ANY of the reported pathways matches
# the true pseudobulk cell type. All datasets use Cell_marker_Human.xlsx as
# the base; brain datasets additionally get Allen Brain Atlas (brain-specific,
# scRNA-derived) on top. Allen's terms use an abbreviated taxonomy
# ("Human Exc ...", "Human Oligo ...") that alias_map's full-word aliases
# ("excitatory neuron") don't match -- match_allen_prefix() below recognizes
# that naming convention specifically (anchored to the literal "Human <Class>"
# prefix, so it can't accidentally match unrelated CellMarker terms).
suppressPackageStartupMessages(library(clusterProfiler))

get_top_genes <- function(Z, lv, top_pct) {
    if (!lv %in% colnames(Z)) return(character(0))
    loadings <- Z[[lv]]; names(loadings) <- rownames(Z)
    n_genes  <- max(1L, ceiling(length(loadings) * top_pct))
    ord      <- order(loadings, decreasing = TRUE)
    names(loadings)[ord][seq_len(n_genes)]
}

parse_gmt <- function(path) {
    lines  <- readLines(path)
    result <- vector("list", length(lines))
    nms    <- character(length(lines))
    for (i in seq_along(lines)) {
        parts       <- strsplit(lines[i], "\t")[[1]]
        nms[i]      <- parts[1]
        result[[i]] <- parts[-(1:2)]
    }
    names(result) <- nms
    result
}

BRAIN_DS <- c("Brain_Mathys2023", "Brain_Xiong2023")

allen_raw  <- parse_gmt(here("data", "pathways", "Allen_Brain_Atlas_10x_scRNA_2021.gmt"))
allen_t2g  <- allen_raw[grepl(" up$", names(allen_raw)) & grepl("^Human ", names(allen_raw))]

brain_t2g <- allen_t2g  # brain = Cell_marker_Human.xlsx (base, cm_t2g) + Allen only

allen_alias_map <- list(
    Ast = "Astro", Exc = "Exc", Inh = "Inh", Mic = "Micro",
    Oli = "Oligo", Opc = "OPC", Vas = c("Endo", "VLMC")
)
match_allen_prefix <- function(cell_type, terms) {
    prefixes <- allen_alias_map[[cell_type]]
    if (is.null(prefixes)) return(rep(FALSE, length(terms)))
    # trailing literal space, not \\b -- R string literals treat \\b as an
    # actual backspace control character, not the 2-char regex escape "\b",
    # so \\b silently never matched anything; Allen's naming always has a
    # space right after the class token ("Human Astro ", not "Human Astrocyte").
    pattern <- paste0("^Human (", paste(prefixes, collapse = "|"), ") ")
    grepl(pattern, terms)
}

TOP_GENE_PCT_FALLBACK <- 0.03  # retried once for LVs with zero significant hits at TOP_GENE_PCT
TOP_N_PATHWAYS        <- 20    # report up to this many significant hits, not just top 5

run_enricher_once <- function(Z, LV, pct, term2gene_full, universe) {
    top_genes <- get_top_genes(Z, LV, pct)
    tryCatch(
        clusterProfiler::enricher(gene = top_genes, universe = universe,
                                  TERM2GENE = term2gene_full, pvalueCutoff = 1,
                                  qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf),
        error = function(e) NULL)
}

run_lv_ora_top5 <- function(ds, LV, term2gene_full, universe) {
    z_path <- file.path(MOD_ROOT, ds, "CLAMPfull", "Z.csv")
    Z      <- read.csv(z_path, row.names = 1, check.names = FALSE)

    res <- run_enricher_once(Z, LV, TOP_GENE_PCT, term2gene_full, universe)
    sig <- if (!is.null(res) && nrow(res@result) > 0) res@result[res@result$p.adjust < 0.05, ] else res@result[0, ]

    # Nothing significant at TOP_GENE_PCT -- retry once at the wider fallback
    # pct before giving up. (Kept separate from the main TOP_GENE_PCT so the
    # 1% default stays untouched for every LV that already works.)
    if (nrow(sig) == 0) {
        res <- run_enricher_once(Z, LV, TOP_GENE_PCT_FALLBACK, term2gene_full, universe)
        sig <- if (!is.null(res) && nrow(res@result) > 0) res@result[res@result$p.adjust < 0.05, ] else res@result[0, ]
    }
    if (nrow(sig) == 0)
        return(data.frame(top5_pathways = NA_character_, top5_fdr = NA_character_))

    topN <- head(sig[order(sig$p.adjust), ], TOP_N_PATHWAYS)
    data.frame(top5_pathways = paste(topN$ID, collapse = ", "),
               top5_fdr      = paste(formatC(topN$p.adjust, format = "e", digits = 2), collapse = ", "))
}

top_pathway_df <- dplyr::bind_rows(lapply(names(module_results), function(ds) {
    universe  <- module_results[[ds]]$universe
    term2gene_src <- if (ds %in% BRAIN_DS) c(cm_t2g, brain_t2g) else cm_t2g  # homogeneous: brain now also includes Cell_marker_Human.xlsx
    src_terms <- names(term2gene_src)
    term2gene_full <- dplyr::bind_rows(lapply(src_terms, function(t) {
        genes <- intersect(term2gene_src[[t]], universe)
        if (length(genes) == 0) return(NULL)
        data.frame(term = t, gene = genes, stringsAsFactors = FALSE)
    }))
    sel_ds <- selected_all_df[selected_all_df$dataset == ds, ]
    dplyr::bind_rows(lapply(seq_len(nrow(sel_ds)), function(i) {
        row <- sel_ds[i, ]
        cbind(row[, c("dataset", "cell_type", "LV")],
              run_lv_ora_top5(ds, row$LV, term2gene_full, universe))
    }))
}))

any_of_top5 <- function(cell_type, top5_str) {
    if (is.na(top5_str)) return(FALSE)
    terms <- strsplit(top5_str, ", ")[[1]]
    any(match_aliases(cell_type, terms) | match_allen_prefix(cell_type, terms))
}
top_pathway_df$recovered <- mapply(any_of_top5, top_pathway_df$cell_type, top_pathway_df$top5_pathways)

top_pathway_table <- top_pathway_df
top_pathway_table$cell_type <- ct_label(top_pathway_table$cell_type)
top_pathway_table <- top_pathway_table[order(top_pathway_table$dataset, top_pathway_table$cell_type), ]
rownames(top_pathway_table) <- NULL

cat("Recovered:", sum(top_pathway_table$recovered), "/", nrow(top_pathway_table), "\n")
top_pathway_table

In [ ]:
# Pooled-marker-set ORA (module_results$marker_sets, the alias-grouped sets),
# used for the recovery dot-grid plot below: rows = every candidate cell type
# in the dataset, giving the full specificity picture per LV. Column labels
# reuse Table 1's single top (rank-1) pathway per LV instead of the coarse
# alias-group name.
build_term2gene <- function(marker_sets) {
    dplyr::bind_rows(lapply(names(marker_sets), function(ct) {
        data.frame(term = ct, gene = marker_sets[[ct]], stringsAsFactors = FALSE)
    }))
}

run_ora_enrichment <- function(selected_lvs_df, Z, term2gene, universe, top_pct) {
    do.call(rbind, lapply(seq_len(nrow(selected_lvs_df)), function(i) {
        row       <- selected_lvs_df[i, ]
        top_genes <- get_top_genes(Z, row$LV, top_pct)
        res <- tryCatch(
            clusterProfiler::enricher(gene = top_genes, universe = universe,
                                      TERM2GENE = term2gene, pvalueCutoff = 1,
                                      qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf),
            error = function(e) NULL)
        if (is.null(res) || nrow(res@result) == 0) return(NULL)
        rdf <- res@result
        data.frame(lv_cell_type = row$cell_type, LV = row$LV,
                  marker_cell_type = rdf$ID, Count = rdf$Count,
                  pvalue = rdf$pvalue, FDR = rdf$p.adjust, stringsAsFactors = FALSE)
    }))
}

run_enrichment_for_datasets <- function(results_list, fun, ...) {
    dplyr::bind_rows(lapply(names(results_list), function(ds) {
        res       <- results_list[[ds]]
        z_path    <- file.path(MOD_ROOT, ds, res$method, "Z.csv")
        Z         <- read.csv(z_path, row.names = 1, check.names = FALSE)
        term2gene <- build_term2gene(res$marker_sets)
        enr <- fun(res$selected, Z, term2gene, res$universe, ...)
        if (is.null(enr) || nrow(enr) == 0) return(NULL)
        cbind(dataset = ds, enr)
    }))
}

enrichment_res <- run_enrichment_for_datasets(module_results, run_ora_enrichment, TOP_GENE_PCT)
# No significance filter and no hardcoded cap on neg_log10_fdr -- size reflects
# the true, continuous -log10(FDR) for every tested pair. A near-miss (e.g.
# Brain_Mathys2023 Excitatory neurons/LV19: own-set FDR = 0.068, one-vs-rest
# effect strongly significant at q=1e-8) still renders as a small-but-real dot
# instead of being deleted outright, while a truly null result (FDR ~ 1, e.g.
# PBMC_1k1k CD8+ T cells/LV57's own set) naturally shrinks to near-invisible
# without needing a hard cutoff. Only genuine zero-overlap pairs (enricher()
# returns nothing at all) are absent here, later filled blank by
# tidyr::complete() in the plot below.
enrichment_res$neg_log10_fdr <- -log10(enrichment_res$FDR + 1e-300)

# Column label: if recovered (any of the up-to-20 candidates matches the true
# cell type via alias_map), show up to the top LABEL_MAX_TERMS matching
# pathways (lowest FDR first), not just one -- e.g. Heart AtrialCM/LV10
# matches both the generic "Cardiomyocyte" (lower FDR) and the chamber-
# specific "Atrial cell" (higher FDR); showing only the generic one hides the
# more informative match. Capped (not "show all matches"): broad categories
# like "Endothelial cells"/"Fibroblasts" can match a dozen+ near-synonymous
# CellMarker terms out of the top-20 candidate pool, and concatenating all of
# them produces multi-hundred-character labels that blow up the plot's
# patchwork layout (panels shrink to nothing, titles overlap). top1_fdr is
# still the single lowest FDR among matches -- unchanged, drives dot color/size.
# Falls back to rank-1 only when nothing matched (recovered == FALSE).
LABEL_MAX_TERMS <- 2
pick_label_hit <- function(cell_type, top5_str, top5_fdr_str) {
    if (is.na(top5_str) || is.na(top5_fdr_str))
        return(data.frame(top1_pathway = NA_character_, top1_fdr = NA_real_))
    paths <- strsplit(top5_str, ", ")[[1]]
    fdrs  <- as.numeric(strsplit(top5_fdr_str, ", ")[[1]])
    matches <- match_aliases(cell_type, paths) | match_allen_prefix(cell_type, paths)
    if (any(matches)) {
        ord   <- order(fdrs[matches])
        keep  <- head(ord, LABEL_MAX_TERMS)
        label <- paste(paths[matches][keep], collapse = " - ")
        fdr   <- min(fdrs[matches])
    } else {
        label <- paths[1]
        fdr   <- fdrs[1]
    }
    data.frame(top1_pathway = label, top1_fdr = fdr, stringsAsFactors = FALSE)
}
top_pathway_hits <- dplyr::bind_rows(mapply(pick_label_hit, top_pathway_df$cell_type,
                                            top_pathway_df$top5_pathways,
                                            top_pathway_df$top5_fdr,
                                            SIMPLIFY = FALSE))
top_pathway_df$top1_pathway <- top_pathway_hits$top1_pathway
top_pathway_df$top1_fdr     <- top_pathway_hits$top1_fdr

# Allen hits are individual scRNA clusters (e.g. "Human Astro L1 FGFR3
# SERPINI2 up") -- too granular for an axis label. Collapse to the cell class
# ("Human Astrocytes") for display only; Table 1's own top5_pathways column
# keeps the exact, unmodified term.
clean_allen_label <- function(x) {
    allen_full_name <- c(Astro = "Astrocytes", Exc = "Excitatory neurons", Inh = "Inhibitory neurons",
                         Micro = "Microglia", Oligo = "Oligodendrocytes", OPC = "OPC",
                         Endo = "Endothelial cells", VLMC = "VLMC")
    pattern  <- "^Human (Astro|Exc|Inh|Micro|Oligo|OPC|Endo|VLMC) "
    is_allen <- grepl(pattern, x)
    cls      <- sub(paste0(pattern, ".*$"), "\\1", x)
    out      <- x
    out[is_allen] <- paste("Human", allen_full_name[cls[is_allen]])
    out
}

# CellMarker_2024.txt terms embed tissue/context text with no fixed delimiter
# (e.g. "Oligodendrocyte Progenitor Cell Embryonic Prefrontal Cortex Human")
# -- strip common anatomical/context words for a readable axis label ("...
# Progenitor Cell Human"), keeping the cell-type description and trailing
# species. Display-only; Table 1\'s own top5_pathways column is untouched.
TISSUE_STOPWORDS <- c(
    "Embryonic", "Prefrontal", "Cortex", "Hippocampus", "Esophagus", "Blood",
    "Peripheral", "Fetal", "Kidney", "Bone", "Marrow", "Spleen", "Nasopharynx",
    "Lung", "Liver", "Skin", "Intestine", "Large", "Small", "Trachea", "Gonad",
    "Undefined", "Tissue", "Adipose", "Brain", "Nasal", "Colon", "Stomach",
    "Pancreatic", "Pancreas", "Islet", "Renal", "Cardiac", "Heart", "Retina",
    "Skeletal", "Muscle", "Ovary", "Testis", "Placenta", "Umbilical", "Cord",
    "Synovial", "Adrenal", "Thyroid", "Salivary", "Mammary", "Prostate",
    "Biliary", "Tract"
)
shorten_cellmarker_label <- function(x) {
    is_cm <- grepl(" (Human|Mouse)$", x) & !is.na(x)
    out <- x
    for (w in TISSUE_STOPWORDS) {
        out <- ifelse(is_cm, gsub(paste0("\\b", w, "\\b\\s*"), "", out), out)
    }
    ifelse(is_cm, trimws(gsub("\\s+", " ", out)), out)
}

# NA (nothing significant at all) shows just the LV code -- not the true cell
# type name, which would misleadingly read as if it were the recovered pathway.
top_pathway_df$top_pathway_label <- ifelse(is.na(top_pathway_df$top1_pathway),
                                            top_pathway_df$LV,
                                            shorten_cellmarker_label(clean_allen_label(top_pathway_df$top1_pathway)))

cat("ORA marker enrichment tests (pooled sets):", nrow(enrichment_res), "\n")

In [ ]:
# Rows = every candidate cell type in the dataset, columns = the LV selected
# per pseudobulk cell type (labeled with Table 1's top-1 pathway). The grid is
# completed to every (row, column) combination -- clusterProfiler::enricher()
# silently omits zero-overlap pairs. FDR/size therefore comes from enrichment
# rows plus recovery overrides, while color/effect is looked up independently
# for every LV x marker-cell pair.
lv_effect_all <- dplyr::bind_rows(lapply(names(module_results), function(ds) {
    res <- module_results[[ds]]
    ovr_lookup <- if (nrow(res$ovr) > 0) res$ovr[, c("cell_type", "LV", "effect")] else
        data.frame(cell_type = character(), LV = character(), effect = numeric())
    names(ovr_lookup) <- c("marker_cell_type", "LV", "ovr_effect")
    corr_lookup <- lv_corr_full[lv_corr_full$dataset == ds, c("LV", "cell_type", "cor")]
    names(corr_lookup) <- c("LV", "marker_cell_type", "corr_effect")

    eff <- merge(corr_lookup, ovr_lookup, by = c("marker_cell_type", "LV"), all = TRUE)
    eff$row_effect <- ifelse(!is.na(eff$ovr_effect), eff$ovr_effect, eff$corr_effect)
    eff <- eff[!is.na(eff$row_effect), c("marker_cell_type", "LV", "row_effect")]
    cbind(dataset = ds, eff)
}))

enr_all <- dplyr::bind_rows(lapply(names(module_results), function(ds) {
    res <- module_results[[ds]]
    enr <- enrichment_res[enrichment_res$dataset == ds, ]
    if (nrow(res$selected) == 0 || nrow(enr) == 0) return(NULL)

    eff <- lv_effect_all[lv_effect_all$dataset == ds,
                         c("dataset", "marker_cell_type", "LV", "row_effect")]
    merge(enr, eff, by = c("dataset", "marker_cell_type", "LV"), all.x = TRUE)
}))

Z_LIM <- max(abs(lv_effect_all$row_effect), na.rm = TRUE)
recovered_top_q <- -log10(top_pathway_df$top1_fdr[top_pathway_df$recovered %in% TRUE] + 1e-300)
Q_LIM <- max(c(enr_all$neg_log10_fdr, recovered_top_q), na.rm = TRUE)

module_panels <- lapply(names(module_results), function(ds) {
    res <- module_results[[ds]]
    sel <- res$selected
    enr <- enr_all[enr_all$dataset == ds, ]
    if (nrow(sel) == 0 || nrow(enr) == 0) return(NULL)

    ct_order <- sort(sel$cell_type)

    tp_ds  <- top_pathway_df[top_pathway_df$dataset == ds, ]
    tp_lut <- setNames(tp_ds$top_pathway_label, tp_ds$cell_type)
    # NA-fallback labels are just the bare LV code (e.g. "LV15") -- don't
    # re-append " - LV15" on top of that or it renders as "LV15 - LV15".
    col_labels <- setNames(
        ifelse(tp_lut[sel$cell_type] == sel$LV, sel$LV, paste0(tp_lut[sel$cell_type], " - ", sel$LV)),
        sel$cell_type)
    # Drop pooled-ORA hits against cell types that never got their own LV
    # (e.g. Lung has 23 candidate cell types but only 17 could be assigned a
    # unique LV out of CLAMPfull's k=18) -- otherwise factor(..., levels=ct_order)
    # silently coerces those out-of-level marker_cell_type values to NA, which
    # tidyr::complete() below then renders as a phantom "NA" row on the plot.
    enr <- enr[enr$marker_cell_type %in% ct_order, ]
    enr$lv_col_label     <- factor(col_labels[enr$lv_cell_type], levels = unname(col_labels[ct_order]))
    enr$marker_row_label <- factor(ct_label(enr$marker_cell_type), levels = ct_label(ct_order))

    # Label-text -> underlying-code lookups, built before tidyr::complete() so
    # marker_cell_type/LV/lv_cell_type (and anything derived from them, like
    # is_diagonal) can be recomputed for rows that completion fills in -- those
    # filled rows only carry the two label columns, everything else comes back NA.
    row_label_to_ct <- setNames(ct_order, ct_label(ct_order))
    col_label_to_lv <- setNames(sel$LV, unname(col_labels[sel$cell_type]))

    # Complete every (row, column) pair -- fill missing (zero-overlap) cells
    # with neg_log10_fdr = 0 (FDR = 1). LV effect is restored below for every
    # completed pair from the independent effect lookup.
    enr <- tidyr::complete(enr, marker_row_label, lv_col_label,
                           fill = list(neg_log10_fdr = 0))
    enr$marker_cell_type <- row_label_to_ct[as.character(enr$marker_row_label)]
    enr$LV               <- col_label_to_lv[as.character(enr$lv_col_label)]
    enr$lv_cell_type      <- sel$cell_type[match(enr$LV, sel$LV)]
    enr$is_diagonal       <- enr$lv_cell_type == enr$marker_cell_type

    enr$row_effect <- NULL
    effect_lookup <- lv_effect_all[lv_effect_all$dataset == ds,
                                   c("marker_cell_type", "LV", "row_effect")]
    enr <- merge(enr, effect_lookup, by = c("marker_cell_type", "LV"), all.x = TRUE)
    enr$row_effect[is.na(enr$row_effect)] <- 0

    # Table 1's recovered full-database ORA hit is the right size evidence for
    # recovered diagonal cells, especially brain Allen terms that are not in the
    # pooled marker-set grid.
    rec_lookup <- top_pathway_df[top_pathway_df$dataset == ds,
                                 c("cell_type", "LV", "recovered", "top1_fdr")]
    names(rec_lookup) <- c("lv_cell_type", "LV", "recovered", "top1_fdr")
    enr <- merge(enr, rec_lookup, by = c("lv_cell_type", "LV"), all.x = TRUE)
    enr$top1_neg_log10_fdr <- -log10(enr$top1_fdr + 1e-300)
    recover_idx <- enr$is_diagonal %in% TRUE & enr$recovered %in% TRUE &
                   !is.na(enr$top1_neg_log10_fdr)
    enr$neg_log10_fdr[recover_idx] <- pmax(enr$neg_log10_fdr[recover_idx],
                                           enr$top1_neg_log10_fdr[recover_idx])
    diag_recovered <- enr[enr$is_diagonal %in% TRUE & enr$recovered %in% TRUE, ]

    n_samples <- nrow(read.csv(file.path(DATA_ROOT, ds, "truthFrac_v0.csv"), row.names = 1, check.names = FALSE))

    ggplot(enr, aes(x = lv_col_label, y = marker_row_label)) +
        geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
        { if (nrow(diag_recovered) > 0)
            geom_point(data = diag_recovered, aes(size = neg_log10_fdr),
                       shape = 1, color = "black", stroke = 1.3) } +
        scale_size_continuous(limits = c(0, Q_LIM), range = c(1, 9),
                              name = expression("-" * log[10] ~ "(FDR)")) +
        scale_color_gradientn(colors = c("#b2182b", "#f4a582", "#f7f7f7", "#a1d99b", "#007a33"),
                              values = c(0, 0.35, 0.5, 0.65, 1),
                              limits = c(-Z_LIM, Z_LIM),
                              name = "LV effect\n(one-vs-rest,\nor r fallback)") +
        scale_x_discrete(drop = FALSE) +
        scale_y_discrete(drop = FALSE) +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = paste0(ds, " (n = ", n_samples, " samples)"))
})
module_panels <- Filter(Negate(is.null), module_panels)

options(repr.plot.width = 20, repr.plot.height = 12)
p_module <- wrap_plots(module_panels, ncol = 3, guides = "collect") &
    theme(legend.position = "right")
print(p_module)

In [ ]:
write.csv(ovr_all_df,        file.path(OUT_DIR, "module_ovr_specificity.csv"),        row.names = FALSE)
write.csv(selected_all_df,   file.path(OUT_DIR, "module_selected_lvs.csv"),           row.names = FALSE)
write.csv(top_pathway_table, file.path(OUT_DIR, "module_top_pathways.csv"),           row.names = FALSE)
# Canonical inputs for 04_hard_cell_types: calculations stay here; that
# notebook only filters and renders this full-data evidence.
write.csv(enr_all,           file.path(OUT_DIR, "marker_enrichment_long.csv"),         row.names = FALSE)
write.csv(lv_effect_all,     file.path(OUT_DIR, "lv_marker_effects.csv"),              row.names = FALSE)
write.csv(top_pathway_df,    file.path(OUT_DIR, "marker_pathway_recovery.csv"),        row.names = FALSE)

cat("Module-level marker recovery outputs saved to:", OUT_DIR, "\n")
ggsave(file.path(OUT_DIR, 'marker_recovery_dotgrid.png'), p_module,
       width = 20, height = 12, dpi = 150, limitsize = FALSE)


## Save outputs

In [ ]:
write.csv(top_lvs_df, file.path(OUT_DIR, "top_lvs_per_celltype.csv"), row.names = FALSE)
write.csv(
  top_lvs_df %>% dplyr::select(dataset, cell_type, LV, cor,
                                r_next_best_ct, r_next_best_lv, margin_ct, margin_lv),
  file.path(OUT_DIR, "lv_specificity_margins.csv"), row.names = FALSE
)

cat("All outputs saved to:", OUT_DIR, "\n")